## 1. Define Global Settings, Parameters and Imports

In [ ]:
import psutil
import pandas as pd
import os
import sys
import tensorflow.compat.v1 as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Filter out INFO and WARNING logs
tf.get_logger().setLevel("ERROR")  # only show error messages


from recommenders.utils.timer import Timer
from recommenders.utils.constants import SEED
from recommenders.models.deeprec.deeprec_utils import prepare_hparams
from recommenders.datasets.amazon_reviews import (
    download_and_extract,
    data_preprocessing,
)
from recommenders.models.deeprec.models.sequential.sli_rec import (
    SLI_RECModel as SeqModel,
)

from recommenders.models.deeprec.io.sequential_iterator import SequentialIterator

# from recommenders.models.deeprec.io.nextitnet_iterator import NextItNetIterator
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"Tensorflow version: {tf.__version__}")

#### Parameters

In [ ]:
EPOCHS = 5
BATCH_SIZE = 400
RANDOM_SEED = SEED  # Set None for non-deterministic result

original_files_path = os.path.join("..", "..", "Data", "original_data")
data_path = os.path.join("..", "..", "Data", "generalization_performance")

##  ATTENTION: change to the corresponding config file, e.g., caser.yaml for CaserModel, sum.yaml for SUMModel
yaml_file = "recommenders/models/deeprec/config/sli_rec.yaml"

#### Amazon Datasets

In [ ]:
output_file = os.path.join(data_path, r"output.txt")

user_vocab = os.path.join(original_files_path, r"user_vocab.pkl")
item_vocab = os.path.join(original_files_path, r"item_vocab.pkl")
cate_vocab = os.path.join(original_files_path, r"category_vocab.pkl")

reviews_name = "Movies_and_TV.jsonl"
meta_name = "meta_Movies_and_TV.jsonl"
reviews_file = os.path.join(original_files_path, reviews_name)
meta_file = os.path.join(original_files_path, meta_name)

train_num_ngs = 4  # number of negative instances with a positive instance for training
valid_num_ngs = (
    4  # number of negative instances with a positive instance for validation
)
test_num_ngs = 9  # number of negative instances with a positive instance for testing

# leaving the original file locations commented in the list as reference to help insert later
input_files = [
    # reviews_file,
    meta_file,
    # train_file,
    # valid_file,
    # test_file,
    user_vocab,
    item_vocab,
    cate_vocab,
]

#### Hyper Parameters

In [ ]:
### Note
### remember to use `_create_vocab(train_file, user_vocab, item_vocab, cate_vocab)` to generate the user_vocab, item_vocab and cate_vocab files, if you are using your own dataset rather than using our demo Amazon dataset.

hparams = prepare_hparams(
    yaml_file,
    embed_l2=0.0,
    layer_l2=0.0,
    learning_rate=0.001,  # set to 0.01 if batch normalization is disable
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    show_step=20,
    MODEL_DIR=os.path.join(data_path, "model/"),
    SUMMARIES_DIR=os.path.join(data_path, "summary/"),
    user_vocab=user_vocab,
    item_vocab=item_vocab,
    cate_vocab=cate_vocab,
    need_sample=True,
    train_num_ngs=train_num_ngs,  # provides the number of negative instances for each positive instance for loss computation.
)

#### Subset Generating Functions

In [ ]:
def generate_dataset(file_prefix, sample_rate, selection_type):

    if not selection_type in ["random", "user", "time"]:
        return

    print("Downloading")
    if not os.path.exists(reviews_file):
        download_and_extract(reviews_name, reviews_file)
        download_and_extract(meta_name, meta_file)

    print("Download complete.")

    print("Splitting beginning.")

    split_file_and_save(
        reviews_file,
        str(sample_rate) + file_prefix,
        sample_rate,
        selection_type,
        (False if selection_type == "random" else True),
    )

    print("Splitting completed.")

    return


# SPLITTING
def split_file_and_save(
    new_reviews_file,
    dest_files_prefix,
    the_sample_rate,
    the_selection_type,
    ignore_test_file=True,
):
    train_file = os.path.join(data_path, dest_files_prefix + r"train_data")
    valid_file = os.path.join(data_path, dest_files_prefix + r"valid_data")
    test_file = os.path.join(data_path, r"IGNORE_test_data")

    if os.path.exists(train_file):
        return

    local_input_files = input_files.copy()
    # insert the new reviews_file
    local_input_files.insert(0, new_reviews_file)

    # Use the test files generated by random file selection
    # only to test the other 2 selection approaches
    if not ignore_test_file:
        test_file = os.path.join(data_path, dest_files_prefix + r"test_data")

    local_input_files.insert(2, train_file)
    local_input_files.insert(3, valid_file)
    local_input_files.insert(4, test_file)

    data_preprocessing(
        *local_input_files,
        the_selection_type,
        sample_rate=the_sample_rate,
        valid_num_ngs=valid_num_ngs,
        test_num_ngs=test_num_ngs
    )

#### Resource Usage Helper Function

In [ ]:
import pickle


def measure_training_resources(model: SeqModel, train_data, val_data, model_save_name):
    """
    Measures memory usage, and time while training.
    """
    # Track memory before starting
    memory_before = psutil.virtual_memory().used

    path_best_trained = os.path.join(hparams.MODEL_DIR, model_save_name) + "_best_model"
    path_best_trained_total_epocs_metrics = (
        os.path.join(hparams.MODEL_DIR, model_save_name) + "_metrics.pkl"
    )
    metrics = None

    if os.path.exists(path_best_trained_total_epocs_metrics):
        model.load_model(path_best_trained)
        with open(path_best_trained_total_epocs_metrics, "rb") as file:
            metrics = pickle.load(file)
    else:
        with Timer() as train_time:
            model, epoch_losses = model.fit(
                train_data,
                val_data,
                valid_num_ngs=valid_num_ngs,
                model_name=model_save_name,
            )

        # Track memory after finishing
        memory_after = psutil.virtual_memory().used

        # Approximate measurements
        memory_usage_change = (memory_after - memory_before) / (1024 * 1024)  # in MB

        # print("Time cost for training is {0:.2f} mins".format(train_time.interval / 60.0))
        metrics = {
            "epoch_losses": epoch_losses,
            "training_time": train_time.interval,
            "memory_usage_change": memory_usage_change,
        }
        with open(path_best_trained_total_epocs_metrics, "wb") as file:
            pickle.dump(metrics, file)

    return metrics

#### Function for saving training results (in case there's a break in execution)

In [ ]:
def append_results(metric, res, subtype, current_frac):
    results_dataframe = None
    if not os.path.exists(data_path + "/RESULTS"):
        df_columns = [
            "subset_type",
            "fraction",
            "train_time",
            "cpu_usage_change",
            "memory_usage_change",
            "test_accuracy",
        ]
        results_dataframe = pd.DataFrame(columns=df_columns)
    else:
        results_dataframe = pd.read_csv(data_path + "/RESULTS")

    # Add the record
    new_record = {
        "subset_type": subtype,
        "fraction": current_frac,
        "train_time": metric["training_time"],
        "cpu_usage_change": metric["cpu_usage_change"],
        "memory_usage_change": metric["memory_usage_change"],
        "test_accuracy": res["auc"],
    }

    results_dataframe.loc[len(results_dataframe)] = new_record

    results_dataframe.to_csv(data_path + "/RESULTS", index=False)


input_creator = SequentialIterator  # Data Loader


def check_if_already_done(subtype, current_frac):
    if not os.path.exists(data_path + "/RESULTS"):
        return False

    saved_results = pd.read_csv(data_path + "/RESULTS")
    if (
        len(
            saved_results[
                (saved_results["subset_type"] == subtype)
                & (saved_results["fraction"] == current_frac)
            ]
        )
        > 0
    ):
        return True
    else:
        return False

## 2. Get Datasets, Train, Evaluate and Chart Results

#### Generate the Sub Datasets (Fractions)

In [ ]:
fractions = [0.25, 0.5, 0.75, 0.9]

for frac in fractions:
    # RANDOM SUBSET
    prefix = "_random_"
    if not os.path.exists(data_path + "/" + str(frac) + prefix + "train_data"):
        generate_dataset(prefix, frac, "random")

    # # USER-BASED SUBSET
    prefix = "_user_"
    if not os.path.exists(data_path + "/" + str(frac) + prefix + "train_data"):
        generate_dataset(prefix, frac, "user")

    # # TIME-BASED SUBSET
    prefix = "_time_"
    if not os.path.exists(data_path + "/" + str(frac) + prefix + "train_data"):
        generate_dataset(prefix, frac, "time")

# FULL DATASET
prefix = "_full_"
if not os.path.exists(data_path + "/" + str(frac) + prefix + "train_data"):
    generate_dataset(prefix, 1.0, "random")

#### Train and Evaluate the Model

In [ ]:
for frac in fractions:
    print(f"---\nExperiment fraction: {frac*100}%")
    test_file = data_path + "/" + str(frac) + "_random_test_data"

    # =========================
    # RANDOM SUBSET
    # =========================
    if not check_if_already_done("Random", frac):
        model_random = SeqModel(hparams, input_creator, seed=RANDOM_SEED)
        model_name = "model_random_" + str(frac)
        metrics_random = measure_training_resources(
            model_random,
            data_path + "/" + str(frac) + "_random_train_data",
            data_path + "/" + str(frac) + "_random_valid_data",
            model_save_name=model_name,
        )

        # Evaluate
        res_syn_random = model_random.run_eval(test_file, num_ngs=test_num_ngs)

        append_results(metrics_random, res_syn_random, "Random", frac)

    # =========================
    # USER-BASED SUBSET
    # =========================
    if not check_if_already_done("User-Based", frac):
        model_user = SeqModel(hparams, input_creator, seed=RANDOM_SEED)
        model_name = "model_user_" + str(frac)
        metrics_user = measure_training_resources(
            model_user,
            data_path + "/" + str(frac) + "_user_train_data",
            data_path + "/" + str(frac) + "_user_valid_data",
            model_save_name=model_name,
        )

        # Evaluate
        res_syn_user = model_user.run_eval(test_file, num_ngs=test_num_ngs)

        append_results(metrics_user, res_syn_user, "User-Based", frac)

    # =========================
    # TIME-BASED SUBSET
    # =========================
    if not check_if_already_done("Time-Based", frac):
        model_time = SeqModel(hparams, input_creator, seed=RANDOM_SEED)
        model_name = "model_time_" + str(frac)
        metrics_time = measure_training_resources(
            model_time,
            data_path + "/" + str(frac) + "_time_train_data",
            data_path + "/" + str(frac) + "_time_valid_data",
            model_save_name=model_name,
        )

        # Evaluate
        res_syn_time = model_time.run_eval(test_file, num_ngs=test_num_ngs)

        append_results(metrics_time, res_syn_time, "Time-Based", frac)

## 3. Generate Charts

In [ ]:
results_df = pd.read_csv(data_path + "/RESULTS")
results_df["fraction"] = pd.to_numeric(results_df["fraction"], errors="coerce")
results_df["train_time"] = pd.to_numeric(results_df["train_time"], errors="coerce")
results_df["train_time"] = results_df["train_time"] * 60 * 60 * 5
results_df["test_accuracy"] = pd.to_numeric(
    results_df["test_accuracy"], errors="coerce"
)
print(results_df)

In [ ]:
import matplotlib.pyplot as plt

# 1. Plot: Train Time vs. Fraction
plt.figure()
pivot_train_time = results_df[["fraction", "subset_type", "train_time"]].pivot(
    index="fraction", columns="subset_type", values="train_time"
)
pivot_train_time.plot(marker="o")  # multiple lines, one per subset_type
plt.title("Training Time by Dataset Fraction and Subset Type")
plt.xlabel("Fraction of Dataset")
plt.ylabel("Training Time (hours)")
plt.legend(title="Subset Type", loc="best")  # Customize legend
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.grid(True, linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.show()

# 2. Plot: Test Accuracy vs. Fraction
plt.figure()
pivot_accuracy = results_df.pivot(
    index="fraction", columns="subset_type", values="test_accuracy"
)
pivot_accuracy.plot(marker="o")
plt.title("AUC Score by Dataset Fraction and Subset Type")
plt.xlabel("Fraction of Dataset")
plt.ylabel("AUC Score")
plt.legend(title="Subset Type", loc="best")  # Customize legend
plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)
plt.grid(True, linestyle="--", linewidth=0.7)
plt.ylim(0.8, 1.0)  # y-axis limits
plt.tight_layout()
plt.show()